# FiftyOne Dataset Viewer and Editor: Workflow Guide

This notebook is designed to load an object detection dataset (COCO format) into FiftyOne, visualize it, edit annotations, and safely shut down the environment. It uses a **non-persistent database workflow** to keep your hard drive clean and ensure you are always loading the freshest data directly from your JSON files.

## 1. Setup and Paths
Start by running the imports (`fiftyone` and `psutil`). Then, define the absolute paths to your images folder and your COCO `.json` file. 
- **Important:** Ensure `dataset_name` is unique for the current session. If it is not, your older dataset with that name will be **deleted** from disk.

## 2. Loading the Dataset
The dataset is loaded using `fo.Dataset.from_dir`. 
- **`persistent = False`**: This tells FiftyOne to load the data into a temporary backend database on disk. When the notebook process ends, this dataset is automatically deleted from FiftyOne's memory.
- **Name Collision Check**: The script automatically checks if a dataset with your chosen name already exists in memory and deletes it first. This allows you to safely re-run the loading cell without errors.

## 3. Visualizing and Editing
Run the `launch_app(dataset, auto=False)` cell to open the FiftyOne UI. 
- You can explore images, filter annotations, and manually edit or add bounding boxes directly in the browser UI.

## 4. Exporting Changes (CRITICAL)
Because this notebook uses `persistent = False`, **any changes made in the FiftyOne UI will be lost when the notebook is closed unless you export them.**
- If you edit bounding boxes, run the **Export Changes** cell.
- This will use `dataset.export()` to generate a brand new COCO `.json` file containing your updated annotations in the specified export directory.

## 5. Safe Shutdown
Always run the final shutdown cell before closing the notebook.
- **`session.close()` / `fo.close_app()`**: This safely terminates the web session and background processes.
- Skipping this step may leave background nodes running, which locks the FiftyOne port (usually `5151`) and prevents you from opening FiftyOne next time. If this happens, use the `psutil` rescue script to kill the stuck processes.

## 6. How to Switch Datasets
To view a different dataset:
1. Export any changes you want to keep.
2. Run the **Safe Shutdown** cell.
3. Update the `images_dir`, `json_path`, and `dataset_name` variables in the setup cell.
4. Run the cells from the top to load the new dataset and launch a new app session.

In [1]:
import fiftyone as fo
import psutil

c:\Users\dennis.dosso\Projects\OBJECT_DETECTION\checkbox_weak_labeler\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Define paths to your images and your COCO JSON
images_dir = r"C:\Users\dennis.dosso\Projects\OBJECT_DETECTION\datasets\checkbox\checkbox.v6i.coco\train"
json_path = r"C:\Users\dennis.dosso\Projects\OBJECT_DETECTION\datasets\checkbox\checkbox.v6i.coco\train\_annotations.checkboxes.coco.json"
dataset_name = "checkbox_dataset"

In [3]:
# 2. Safely load the dataset into FiftyOne
if fo.dataset_exists(dataset_name):
    print(f"Dataset '{dataset_name}' already exists. Deleting it to reload fresh data...")
    fo.delete_dataset(dataset_name)

dataset = fo.Dataset.from_dir(
    dataset_type=fo.types.COCODetectionDataset,
    data_path=images_dir,
    labels_path=json_path,
    name=dataset_name
)
dataset.persistent = False
print(f"Dataset '{dataset_name}' loaded successfully!")

 100% |█████████████████| 111/111 [1.0s elapsed, 0s remaining, 107.1 samples/s]         
Dataset 'checkbox_dataset' loaded successfully!


In [4]:
# 3. Launch the FiftyOne App
# Setting auto=False prevents it from opening a new browser tab every time you run the cell
session = fo.launch_app(dataset, auto=False)


Could not connect session, trying again in 10 seconds

Session launched. Run `session.show()` to open the App in a cell output.


███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.20.1

🤖  Ask the Docs Agent       https://docs.voxel51.com
🚀  Getting Started Guides   https://docs.voxel51.com/getting_started
💬  Join the Community       https://community.voxel51.com
🎓  Book a Workshop          https://voxel51.com/workshops



### Export Changes

Run this cell ONLY after you have made edits in the FiftyOne UI and want to save them.

In [ ]:
# 4. Save your edits back to a COCO JSON
export_dir = r"C:\Users\dennis.dosso\Projects\OBJECT_DETECTION\datasets\checkbox\checkbox.v6i.coco\train_edited"

dataset.export(
    export_dir=export_dir,
    dataset_type=fo.types.COCODetectionDataset,
    label_field="ground_truth", # Default field for COCO labels in FiftyOne
)
print("Changes successfully exported!")

In [5]:
# 5. Safely close everything
if 'session' in locals() and session is not None:
    try:
        session.close()
        print(" - Sessione disconnessa.")
    except Exception as e:
        print(f" - Errore durante la disconnessione della sessione: {e}")

try:
    fo.close_app()
    print(" - Processi web terminati.")
except Exception as e:
    pass

print("Chiusura completata. Arrivederci!")

 - Sessione disconnessa.
 - Processi web terminati.
Chiusura completata. Arrivederci!


In [6]:
# Optional rescue cell if the port gets stuck
for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
    try:
        if proc.info['cmdline'] and 'fiftyone' in ' '.join(proc.info['cmdline']):
            proc.kill()
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass